In [0]:
from pyspark.sql.functions import col, from_unixtime, to_date, trim, when, lit

# Define parameters via widgets
dbutils.widgets.text("catalog", "dbr_dev", "1. Catalog Name")
dbutils.widgets.text("bronze_schema", "valeriimatviiv_bronze", "2. Bronze Schema")
dbutils.widgets.text("silver_schema", "valeriimatviiv_silver", "3. Silver Schema")

catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")

source_table = f"{catalog}.{bronze_schema}.finnhub_news_bronze"
target_table = f"{catalog}.{silver_schema}.finnhub_news_silver"

# 1. Read Bronze News Data
df_bronze = spark.read.table(source_table)

# 2. Clean text, convert timestamps, map blank/null symbols to GENERAL
df_silver = (
    df_bronze
    .withColumn("ArticleId", col("id").cast("string"))
    .withColumn("NewsTimestamp", from_unixtime(col("datetime").cast("long")).cast("timestamp"))
    .withColumn("NewsDate", to_date(from_unixtime(col("datetime").cast("long"))))
    .withColumn("Headline", trim(col("headline")))
    .withColumn("Summary", trim(col("summary")))
    # Handle both NULL and empty string "" values for related symbol
    .withColumn(
        "Symbol",
        when((col("related").isNull()) | (trim(col("related")) == ""), lit("GENERAL"))
        .otherwise(trim(col("related")))
    )
    .select(
        "ArticleId",
        "Symbol",
        "NewsDate",
        "NewsTimestamp",
        "Headline",
        "Summary",
        "url",
        "source",
        "category",
        "_schema_phase",
        "_source_file",
        "_ingest_timestamp"
    )
    .dropna(subset=["ArticleId", "Headline", "NewsDate"])
    .dropDuplicates(["ArticleId"])
    .orderBy("NewsDate", "Symbol")
)

# 3. Write Cleaned Data to Silver Delta Table
df_silver.write.format("delta").mode("overwrite").saveAsTable(target_table)

print(f"Successfully processed Silver News table: '{target_table}'")

In [0]:
# catalog = dbutils.widgets.get("catalog")
# silver_schema = dbutils.widgets.get("silver_schema")
# target_table = f"{catalog}.{silver_schema}.finnhub_news_silver"

# df_silver_news = spark.read.table(target_table)

# print("--- Symbol Breakdown ---")
# display(df_silver_news.groupBy("Symbol").count())

# print("--- Sample Preview ---")
# display(df_silver_news.select("ArticleId", "Symbol", "_schema_phase", "NewsDate", "Headline").limit(10))